# Lecture 9 — QSVT & Matrix Functions (UnitaryLab companion)

The **quantum singular value transformation (QSVT)** lifts QSP from a scalar
signal to the singular values of a block-encoded matrix: given a block encoding of
$A$, QSVT applies a polynomial $p$ to each singular value, realizing $p(A)$ (for
Hermitian $A$, a polynomial of its eigenvalues). Choosing $p(x) \approx 1/x$ on the
spectrum of $A$ turns QSVT into a **linear-system solver**: it prepares a state
proportional to $A^{-1}b$.

`ua.QSVTLinearSolverAlgorithm().run(A, b, epsilon)` does exactly this. This
notebook solves a small system and verifies the returned solution against
`numpy.linalg.solve`, both in direction (fidelity) and — because the algorithm
returns a rescaled vector — in absolute value after undoing the scaling.

Contents:
1. Setup
2. Introspect the QSVT result dictionary
3. Verify $x \propto A^{-1}b$ (direction fidelity)
4. Absolute agreement via the reported scaling factor
5. A second system
6. Your turn

In [1]:
import numpy as np
import unitarylab_algorithms as ua
from unitarylab.core import Circuit
import contextlib, io

# Suppress the algorithms' verbose logging so cell outputs stay clean.
@contextlib.contextmanager
def quiet():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yield

np.set_printoptions(precision=4, suppress=True)
print("ready: unitarylab_algorithms loaded")

ready: unitarylab_algorithms loaded


## 2. Introspect the QSVT result dictionary

The solver returns the `Solution vector`, a `Scaling factor applied`, and timing.
We inspect the keys and values before comparing to the classical solution.

In [2]:
A = np.array([[2, 1], [1, 2]], dtype=complex)      # Hermitian, well-conditioned
b = np.array([1, 0], dtype=complex)
epsilon = 0.01

with quiet():
    res = ua.QSVTLinearSolverAlgorithm().run(A, b, epsilon, backend="numpy")
print("keys:", list(res.keys()))
print("Solution vector      :", np.round(res["Solution vector"], 5))
print("Scaling factor applied:", round(float(res["Scaling factor applied"]), 5))

keys: ['status', 'circuit_path', 'plot', 'circuit', 'Solution vector', 'Scaling factor applied', 'Simulation time (s)']
Solution vector      : [ 0.6667+0.j -0.3333+0.j]
Scaling factor applied: 6.0


## 3. Verify $x \propto A^{-1}b$ (direction fidelity)

A quantum linear-system solver returns the solution *direction* — the normalized
state $A^{-1}b / \|A^{-1}b\|$. We compare the normalized QSVT output to the
normalized classical solution via the fidelity $|\langle x_q | x_c\rangle|^2$,
which must be essentially 1.

In [3]:
normalize = lambda v: v / np.linalg.norm(v)
x_q = res["Solution vector"]
x_c = np.linalg.solve(A, b)
fidelity = abs(np.vdot(normalize(x_q), normalize(x_c))) ** 2
print("QSVT  (normalized):", np.round(normalize(x_q), 5))
print("numpy (normalized):", np.round(normalize(x_c), 5))
print("direction fidelity :", round(fidelity, 8))
assert fidelity > 1 - 1e-4
print("Verified: QSVT solution direction matches A^{-1}b (fidelity > 1 - 1e-4).")

QSVT  (normalized): [ 0.8944+0.j -0.4472+0.j]
numpy (normalized): [ 0.8944+0.j -0.4472+0.j]
direction fidelity : 1.0
Verified: QSVT solution direction matches A^{-1}b (fidelity > 1 - 1e-4).


## 4. Absolute agreement via the reported scaling factor

QSVT block-encodes $c/x$ for some constant $c$, so the raw amplitudes are scaled.
The algorithm reports the `Scaling factor applied`; the returned `Solution vector`
already matches $A^{-1}b$ to within the requested $\epsilon$. We check the full
vector (not just its direction) at a tolerance tied to $\epsilon$.

In [4]:
err = np.linalg.norm(x_q - x_c)
print("QSVT solution  :", np.round(x_q, 5))
print("numpy solution :", np.round(x_c, 5))
print("L2 error       :", format(err, ".2e"), " (requested epsilon =", epsilon, ")")
assert err < 10 * epsilon
print("Verified: |x_QSVT - x_numpy| within a constant factor of epsilon.")

QSVT solution  : [ 0.6667+0.j -0.3333+0.j]
numpy solution : [ 0.6667+0.j -0.3333+0.j]
L2 error       : 2.69e-07  (requested epsilon = 0.01 )
Verified: |x_QSVT - x_numpy| within a constant factor of epsilon.


## 5. A second system

To show the assert is not tuned to one matrix, solve a different Hermitian
system and repeat the fidelity check.

In [5]:
A2 = np.array([[3, -1], [-1, 2]], dtype=complex)
b2 = np.array([1, 1], dtype=complex)
with quiet():
    res2 = ua.QSVTLinearSolverAlgorithm().run(A2, b2, 0.01, backend="numpy")
xq2, xc2 = res2["Solution vector"], np.linalg.solve(A2, b2)
fid2 = abs(np.vdot(normalize(xq2), normalize(xc2))) ** 2
print("kappa(A2)         =", round(np.linalg.cond(A2), 4))
print("direction fidelity =", round(fid2, 8))
assert fid2 > 1 - 1e-4
print("Verified: second QSVT solve matches numpy (fidelity > 1 - 1e-4).")

kappa(A2)         = 2.618
direction fidelity = 1.0
Verified: second QSVT solve matches numpy (fidelity > 1 - 1e-4).


## 6. Your turn
- Solve a system whose matrix is closer to singular (small eigenvalue) and watch
  the accuracy degrade at fixed $\epsilon$ — QSVT cost grows with the condition
  number $\kappa$.
- Tighten $\epsilon$ and confirm the L2 error shrinks accordingly.
- Compare against the HHL solver of Lecture 10 on the same matrix.

See Lecture 9 for the general QSVT recipe: even/odd polynomials, the sign and
threshold functions, and matrix functions beyond $1/x$.